# 10 · Build and train a tiny vision transformer

**Level:** advanced. **Prerequisites:** CNNs, tensor shapes, training, and validation.
**Live demonstration:** approximately 45–60 minutes, plus training/download time.

Turn an image into patches, project patches into tokens, add positions and a class token,
train a transformer classifier, and compare it with a CNN using the same data split.
Everything needed for the default experiment trains locally; no pretrained download is required.
The tiny model is a teaching architecture, not a reproduction of a large benchmark ViT.


## 1. Configure a small experiment

Run the environment setup notebook first. CUDA is used when available; CPU is supported.
`QUICK=True` keeps this lesson small enough for a live class. Larger training runs should
be prepared before the presentation, and results should be labeled with their actual configuration.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


In [ ]:
import math
import time
from torch import nn
from torchvision import transforms

cu.seed_everything(42)
N_TRAIN = 128 if SMOKE else (4000 if QUICK else 20000)
N_VAL = 48 if SMOKE else (800 if QUICK else 3000)
N_TEST = 48 if SMOKE else (1000 if QUICK else 3000)
EPOCHS = 1 if SMOKE else (3 if QUICK else 10)
IMAGE_SIZE = 28
PATCH_SIZE = 7
if not isinstance(PATCH_SIZE, int) or PATCH_SIZE <= 0 or IMAGE_SIZE % PATCH_SIZE:
    raise ValueError('PATCH_SIZE must be a positive integer that divides IMAGE_SIZE=28')
GRID_SIDE = IMAGE_SIZE // PATCH_SIZE
NUM_PATCHES = GRID_SIDE ** 2
EMBED_DIM = 64
HEADS = 4
LAYERS = 2


## 2. Use one split for both models

The CNN and transformer will see the same training, validation, and test examples.
We compare a short shared training budget, not fully tuned architectures.
The test set stays untouched until the comparison protocol is fixed.


In [ ]:
train_ds, val_ds, test_ds, class_names = cu.fashion_data(
    train_transform=transforms.ToTensor(), eval_transform=transforms.ToTensor(),
    train_limit=N_TRAIN, val_limit=N_VAL, test_limit=N_TEST, seed=42,
)
val_loader = cu.loader(val_ds)
example_image, example_label = train_ds[0]
print('Image shape [channels, height, width]:', tuple(example_image.shape))
print('Class:', class_names[example_label])
print('Split sizes:', len(train_ds), len(val_ds), len(test_ds))


## 3. Cut the image into patches

With the default settings, a $28\times28$ image with $7\times7$ non-overlapping patches produces
$(28/7)^2=16$ patches. Each grayscale patch contains $1\times7\times7=49$ values.
`unfold` exposes a sliding-window view; here the stride equals the window size.
The plot and shape checks adapt when `PATCH_SIZE` changes to another divisor of 28.

**Ask the audience:** what happens to the number of tokens if the patch side is halved?
For divisible image sizes, there are four times as many patches.


In [ ]:
patch_grid = example_image.unfold(1, PATCH_SIZE, PATCH_SIZE).unfold(2, PATCH_SIZE, PATCH_SIZE)
# [C, rows, columns, patch_h, patch_w] -> [number_of_patches, flattened_patch]
patches = patch_grid.permute(1, 2, 0, 3, 4).reshape(-1, PATCH_SIZE * PATCH_SIZE)
print('Patch grid:', tuple(patch_grid.shape))
print('Flattened patches:', tuple(patches.shape))
figure_side = min(10, max(3.5, GRID_SIDE * 1.2))
fig, axes = plt.subplots(GRID_SIDE, GRID_SIDE, figsize=(figure_side, figure_side), squeeze=False)
for index, ax in enumerate(axes.flat):
    ax.imshow(patches[index].reshape(PATCH_SIZE, PATCH_SIZE), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Patch {index}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()


## 4. Project patches into learned token vectors

A convolution with `kernel_size=patch_size` and `stride=patch_size` implements a shared
linear patch projection efficiently. Its output is flattened into `[batch, tokens, embedding]`.
Each token here has 64 learned features instead of 49 raw pixel values.


In [ ]:
patch_projection = nn.Conv2d(1, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
batch = example_image.unsqueeze(0)
projected = patch_projection(batch)
tokens = projected.flatten(2).transpose(1, 2)
print('Projected feature grid:', tuple(projected.shape))
print('Patch tokens [B, N, D]:', tuple(tokens.shape))
assert tokens.shape == (1, NUM_PATCHES, EMBED_DIM)


## 5. Understand a single attention head

Learned projections produce queries $Q$, keys $K$, and values $V$.
Weights are $\operatorname{softmax}(QK^T/\sqrt{d_k})$; multiplying by $V$ combines
information across tokens. The following randomly initialized head demonstrates shape
and normalization only. Its weights are **not** a trained explanation.


In [ ]:
head_dim = EMBED_DIM // HEADS
q_projection = nn.Linear(EMBED_DIM, head_dim)
k_projection = nn.Linear(EMBED_DIM, head_dim)
v_projection = nn.Linear(EMBED_DIM, head_dim)
q, k, v = q_projection(tokens), k_projection(tokens), v_projection(tokens)
attention = (q @ k.transpose(-2, -1) / math.sqrt(head_dim)).softmax(dim=-1)
mixed = attention @ v
print('Q, K, V:', tuple(q.shape), tuple(k.shape), tuple(v.shape))
print('Attention:', tuple(attention.shape), 'Mixed values:', tuple(mixed.shape))
print('Attention row sums:', attention[0].sum(-1).detach())
plt.figure(figsize=(5, 4))
plt.imshow(attention[0].detach(), cmap='viridis')
plt.xlabel('Key patch')
plt.ylabel('Query patch')
plt.title('One untrained attention head')
plt.colorbar(label='Attention weight')
plt.tight_layout()
plt.show()


## 6. Assemble the tiny ViT

The learned class token collects information for classification. Learned position
embeddings identify patch locations. Each encoder block contains multi-head self-attention,
a feed-forward network, residual connections, and layer normalization.

We construct blocks independently in a `ModuleList`, so each block gets its own random
initialization. `batch_first=True` preserves `[B, N, D]`; `norm_first=True` uses pre-normalization.
The classifier returns logits and must not apply softmax before cross-entropy.


In [ ]:
class TinyViT(nn.Module):
    def __init__(self, image_size=28, patch_size=7, in_channels=1,
                 num_classes=10, embed_dim=64, heads=4, depth=2, dropout=0.1):
        super().__init__()
        if not isinstance(patch_size, int) or patch_size <= 0 or image_size % patch_size:
            raise ValueError('image_size must be divisible by patch_size')
        if embed_dim % heads:
            raise ValueError('embed_dim must be divisible by heads')
        self.image_size = image_size
        self.in_channels = in_channels
        self.patch_embed = nn.Conv2d(in_channels, embed_dim,
                                     kernel_size=patch_size, stride=patch_size)
        self.num_patches = (image_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.position = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=embed_dim, nhead=heads, dim_feedforward=embed_dim * 4,
                dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
            ) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.position, std=0.02)

    def forward(self, x):
        if tuple(x.shape[1:]) != (self.in_channels, self.image_size, self.image_size):
            raise ValueError('Unexpected channel count or image dimensions')
        patch_tokens = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        sequence = torch.cat([cls_tokens, patch_tokens], dim=1)
        sequence = self.dropout(sequence + self.position)
        for block in self.blocks:
            sequence = block(sequence)
        return self.head(self.norm(sequence)[:, 0])

cu.seed_everything(42)
vit = TinyViT(image_size=IMAGE_SIZE, patch_size=PATCH_SIZE, embed_dim=EMBED_DIM,
              heads=HEADS, depth=LAYERS).to(DEVICE)
with torch.no_grad():
    output = vit(torch.stack([train_ds[0][0], train_ds[1][0]]).to(DEVICE))
print('Logits shape:', tuple(output.shape))
print('Sequence length including class token:', vit.num_patches + 1)
print('Trainable parameters:', sum(p.numel() for p in vit.parameters() if p.requires_grad))
assert output.shape == (2, len(class_names))


## 7. Train the transformer

Both models use the same seed, training split, batch size, epochs, learning rate, and
weight decay. This is a controlled classroom comparison, not an equal-compute benchmark.
The validation loss selects the checkpoint separately for each model.


In [ ]:
def synchronize_device():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

# Reset before initialization: the preceding shape demo must not change the experiment.
cu.seed_everything(42)
vit = TinyViT(image_size=IMAGE_SIZE, patch_size=PATCH_SIZE, embed_dim=EMBED_DIM,
              heads=HEADS, depth=LAYERS).to(DEVICE)
vit_loader = cu.loader(train_ds, batch_size=64, shuffle=True, seed=42)
synchronize_device()
start = time.perf_counter()
vit_history = cu.fit(vit, vit_loader, val_loader, EPOCHS, DEVICE,
                     lr=1e-3, weight_decay=1e-4)
synchronize_device()
vit_seconds = time.perf_counter() - start
cu.plot_history(vit_history)
print(f'Transformer training time: {vit_seconds:.1f} seconds')


## 8. Train the CNN baseline

Convolutions encode locality and shared filters directly. A transformer has different
inductive biases and can need more data or stronger pretraining. A larger architecture
is not guaranteed to win a short, small-data experiment.


In [ ]:
cu.seed_everything(42)
cnn = cu.SmallCNN(num_classes=len(class_names), in_channels=1).to(DEVICE)
cnn_loader = cu.loader(train_ds, batch_size=64, shuffle=True, seed=42)
synchronize_device()
start = time.perf_counter()
cnn_history = cu.fit(cnn, cnn_loader, val_loader, EPOCHS, DEVICE,
                     lr=1e-3, weight_decay=1e-4)
synchronize_device()
cnn_seconds = time.perf_counter() - start
cu.plot_history(cnn_history)
print(f'CNN training time: {cnn_seconds:.1f} seconds')


## 9. Compare validation results before opening the test set

Report parameter counts and observed training time alongside accuracy. Time depends on
hardware, software, other processes, and warm-up. These measurements are descriptive.
The two models are predeclared comparisons; do not change their configuration after seeing test results.


In [ ]:
import pandas as pd

models = {'Tiny ViT': vit, 'Small CNN': cnn}
training_times = {'Tiny ViT': vit_seconds, 'Small CNN': cnn_seconds}
validation_rows = []
for name, candidate in models.items():
    metrics = cu.evaluate(candidate, val_loader, DEVICE)
    validation_rows.append({
        'model': name,
        'parameters': sum(p.numel() for p in candidate.parameters()),
        'training_seconds': training_times[name],
        'validation_loss': metrics['loss'],
        'validation_accuracy': metrics['accuracy'],
    })
validation_table = pd.DataFrame(validation_rows)
display(validation_table.round(4))


## 10. Evaluate both frozen models once

No model selection or hyperparameter changes follow this step. A single seed and a small
test subset do not establish a general ranking of CNNs and transformers.


In [ ]:
test_loader = cu.loader(test_ds)
test_rows = []
test_predictions = {}
for name, candidate in models.items():
    test_logits, test_labels = cu.predict(candidate, test_loader, DEVICE)
    test_predictions[name] = test_logits.argmax(dim=1)
    test_rows.append({
        'model': name,
        'test_loss': float(nn.functional.cross_entropy(test_logits, test_labels)),
        'test_accuracy': float((test_logits.argmax(1) == test_labels).float().mean()),
    })
test_table = pd.DataFrame(test_rows)
display(test_table.round(4))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, ax in enumerate(axes.flat):
    image, label = test_ds[idx]
    ax.imshow(image[0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'True: {class_names[label]}\n'
                 f'ViT: {class_names[int(test_predictions["Tiny ViT"][idx])]}\n'
                 f'CNN: {class_names[int(test_predictions["Small CNN"][idx])]}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()
cu.ARTIFACTS.mkdir(parents=True, exist_ok=True)
validation_table.merge(test_table, on='model').to_csv(cu.ARTIFACTS / '10_vit_cnn_comparison.csv', index=False)


## 11. Optional: run an ImageNet-pretrained ViT on your own photograph

This cell is disabled by default. Set `RUN_PRETRAINED=True` and provide an existing photo path.
The first enabled run downloads approximately 330 MB of weights. Use batch size one on
an 8 GB GPU. These weights predict **ImageNet classes**, not the ten Fashion-MNIST classes.
The exact pretrained weights provide their own resizing, cropping, and normalization
through `weights.transforms()`; do not reuse this lesson's bare `ToTensor` transform.
The official [ViT-B/16 documentation](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vit_b_16.html)
describes the weights and preprocessing.


In [ ]:
RUN_PRETRAINED = False
PHOTO_PATH = Path('my_photo.jpg')  # Replace with an absolute path to your RGB photograph.

if RUN_PRETRAINED and not SMOKE:
    from PIL import Image
    from torchvision.models import vit_b_16, ViT_B_16_Weights

    if not PHOTO_PATH.is_file():
        raise FileNotFoundError(f'Choose an existing photo first: {PHOTO_PATH.resolve()}')
    weights = ViT_B_16_Weights.IMAGENET1K_V1
    pretrained = vit_b_16(weights=weights).to(DEVICE).eval()
    photo = Image.open(PHOTO_PATH).convert('RGB')
    photo_tensor = weights.transforms()(photo).unsqueeze(0).to(DEVICE)
    print('Preprocessed shape:', tuple(photo_tensor.shape))
    with torch.inference_mode():
        confidence, indices = pretrained(photo_tensor).softmax(1)[0].topk(5)
    for value, index in zip(confidence.cpu(), indices.cpu()):
        print(f'{weights.meta["categories"][int(index)]:25s} {float(value):.3f}')
    plt.figure(figsize=(5, 4))
    plt.imshow(photo)
    plt.axis('off')
    plt.show()
    del pretrained
else:
    print('Optional pretrained photo inference skipped.')


## 12. Exercises and expert extensions

1. Compare `PATCH_SIZE=4`, `7`, and `14` on validation. Change the setting in Step 1,
   restart the kernel, and run only through Step 9 for each candidate. Choose the final
   configuration before opening a fresh final test set; keep test results out of selection.
2. Remove position embeddings and repeat a validation experiment. What information is lost?
3. Compare two training-set sizes while keeping the validation subset fixed.
4. For a custom dataset, describe how to replace the pretrained classification head and
   avoid leaking related photographs across splits.

<details><summary>Suggested solutions and discussion</summary>

For a 28-pixel image, patch sizes 4, 7, and 14 produce 49, 16, and 4 patches, respectively;
add one class token in every case. Attention matrices scale quadratically with sequence
length. Smaller patches can therefore increase memory and computation substantially.
Without positions, the class-token output cannot directly distinguish permutations of
the patch tokens in this architecture. For pretrained torchvision ViT-B/16, replace
`pretrained.heads.head` with `nn.Linear(pretrained.heads.head.in_features, num_classes)`;
the replacement is untrained and needs supervised training. Start with a frozen backbone,
then consider unfreezing using a small learning rate. Split by subject or source when
images from the same source are related, and validate before final testing.

</details>

**Demonstration checkpoint:** explain `[B, NUM_PATCHES, EMBED_DIM]`, then the added class
token, then `[B, 10]`. With the defaults these are `[B, 16, 64]`, `[B, 17, 64]`, and `[B, 10]`;
train both models and describe the limits of the comparison using the observed results.


## References and next lesson

- [Original Vision Transformer paper](https://arxiv.org/abs/2010.11929)
- [PyTorch TransformerEncoderLayer](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html)
- [Torchvision ViT-B/16 weights and transforms](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vit_b_16.html)

Next: [11 · Robustness and calibration](11_robustness_and_calibration.ipynb).
